# Custom Chatbot Project

TODO: In this cell, write an explanation of which dataset you have chosen and why it is appropriate for this task

## Data Wrangling

TODO: In the cells below, load your chosen dataset into a `pandas` dataframe with a column named `"text"`. This column should contain all of your text data, separated into at least 20 rows.

In [1]:
import openai
import pandas as pd
import re
from dateutil.parser import parse
openai.api_base = "https://openai.vocareum.com/v1"
openai.api_key = "YOUR-API-KEY"

In [29]:
df = pd.read_csv('data/2023_fashion_trends.csv')
# Combine relevant columns into a single "text" column
print(df.columns)
df["text"] = df.apply(
    lambda row: f"{row['Trends']} , {row['Source']}",
    axis=1
)
df = df[(df["text"].str.len() > 0) & (~df["text"].str.startswith("=="))]


# Show the first few rows to verify
print(df[["text"]].head())
print(f"Number of rows: {len(df)}")

'''
I have chosen the 2023_fashion_trends.csv dataset, which contains curated reports and expert quotes about fashion trends observed in 2023. Each entry includes the source URL, article title, and a text snippet highlighting specific trends or designer commentary. This dataset is appropriate for building a fashion-focused chatbot because it provides up-to-date, domain-specific knowledge that is unlikely to be fully captured in a general-purpose language model, especially for recent years. A custom chatbot built on this data can accurately answer questions about 2023 fashion trends, designer influences, and prevailing industry themes—making it a useful tool for fashion students, journalists, or enthusiasts who want authoritative, current information
'''

Index(['URL', 'Trends', 'Source'], dtype='object')
                                                text
0  2023 Fashion Trend: Red. Glossy red hues took ...
1  2023 Fashion Trend: Cargo Pants. Utilitarian w...
2  2023 Fashion Trend: Sheer Clothing. "Bare it a...
3  2023 Fashion Trend: Denim Reimagined. From dou...
4  2023 Fashion Trend: Shine For The Daytime. The...
Number of rows: 82


'\nI have chosen the 2023_fashion_trends.csv dataset, which contains curated reports and expert quotes about fashion trends observed in 2023. Each entry includes the source URL, article title, and a text snippet highlighting specific trends or designer commentary. This dataset is appropriate for building a fashion-focused chatbot because it provides up-to-date, domain-specific knowledge that is unlikely to be fully captured in a general-purpose language model, especially for recent years. A custom chatbot built on this data can accurately answer questions about 2023 fashion trends, designer influences, and prevailing industry themes—making it a useful tool for fashion students, journalists, or enthusiasts who want authoritative, current information\n'

In [23]:
!ls

data  project.ipynb


## Custom Query Completion

TODO: In the cells below, compose a custom query using your chosen dataset and retrieve results from an OpenAI `Completion` model. You may copy and paste any useful code from the course materials.

In [3]:
import tiktoken
# Create a tokenizer that is designed to align with our embeddings
tokenizer = tiktoken.get_encoding("cl100k_base")

#token_limit = 1000
#USER_QUESTION = """Is there any australian?"""

In [4]:
EMBEDDING_MODEL_NAME = "text-embedding-ada-002"
batch_size = 200
embeddings = []
for i in range(0, len(df), batch_size):
    # Send text data to OpenAI model to get embeddings
    response = openai.Embedding.create(
        input=df.iloc[i:i+batch_size]["text"].tolist(),
        engine=EMBEDDING_MODEL_NAME
    )
    
    # Add embeddings to list
    embeddings.extend([data["embedding"] for data in response["data"]])

# Add embeddings list to dataframe
df["embeddings"] = embeddings
df

,URL,Trends,Source,text,embeddings
0,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Red. Glossy red hues took ...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend: Red. Glossy red hues took ...,"[-0.014944399707019329, -0.019510744139552116,..."
1,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Cargo Pants. Utilitarian w...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend: Cargo Pants. Utilitarian w...,"[-0.00013931268767919391, -0.03050145506858825..."
2,https://www.refinery29.com/en-us/fashion-trend...,"2023 Fashion Trend: Sheer Clothing. ""Bare it a...",7 Fashion Trends That Will Take Over 2023 — Sh...,"2023 Fashion Trend: Sheer Clothing. ""Bare it a...","[-0.007677432149648666, -0.021829167380928993,..."
3,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Denim Reimagined. From dou...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend: Denim Reimagined. From dou...,"[-0.010241450741887093, -0.01035370398312807, ..."
4,https://www.refinery29.com/en-us/fashion-trend...,2023 Fashion Trend: Shine For The Daytime. The...,7 Fashion Trends That Will Take Over 2023 — Sh...,2023 Fashion Trend: Shine For The Daytime. The...,"[-0.0012267997954040766, 0.0007112607127055526..."
...,...,...,...,...,...
77,https://www.whowhatwear.com/spring-summer-2023...,"If lime green isn't your vibe, rest assured th...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,"If lime green isn't your vibe, rest assured th...","[0.0012748254230245948, -0.018452078104019165,..."
78,https://www.whowhatwear.com/spring-summer-2023...,"""As someone who can clearly (not fondly) remem...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,"""As someone who can clearly (not fondly) remem...","[-0.008389304392039776, -0.005167033523321152,..."
79,https://www.whowhatwear.com/spring-summer-2023...,"""Combine this design shift with the fact that ...",Spring/Summer 2023 Fashion Trends: 21 Expert-A...,"""Combine this design shift with the fact that ...","[-0.016533413901925087, -0.022308049723505974,..."
80,https://www.whowhatwear.com/spring-summer-2023...,Thought party season ended at the stroke of mi...,Spring/Summer 2023 Fashion Trends: 21 Expert-A...,Thought party season ended at the stroke of mi...,"[-0.014681827276945114, -0.019859733060002327,..."


In [5]:

from openai.embeddings_utils import get_embedding, distances_from_embeddings

def get_rows_sorted_by_relevance(question, df):
    """
    Function that takes in a question string and a dataframe containing
    rows of text and associated embeddings, and returns that dataframe
    sorted from least to most relevant for that question
    """
    
    # Get embeddings for the question text
    question_embeddings = get_embedding(question, engine=EMBEDDING_MODEL_NAME)
    
    # Make a copy of the dataframe and add a "distances" column containing
    # the cosine distances between each row's embeddings and the
    # embeddings of the question
    df_copy = df.copy()
    df_copy["distances"] = distances_from_embeddings(
        question_embeddings,
        df_copy["embeddings"].values,
        distance_metric="cosine"
    )
    
    # Sort the copied dataframe by the distances and return it
    # (shorter distance = more relevant so we sort in ascending order)
    df_copy.sort_values("distances", ascending=True, inplace=True)
    return df_copy

In [19]:
def build_prompt(user_question, df, tokenizer, token_limit=1000):
    prompt_template = """
Answer the question based on the context below, if you do not have enough context, reply with:
"I don't know"

Context: 

{}

---

Question: {}
Answer:"""
    # Start by counting tokens for the user question and prompt template
    token_count = len(tokenizer.encode(user_question)) + len(tokenizer.encode(prompt_template))
    context_list = []
    
    # Add context rows until we reach the token limit
    for text in get_rows_sorted_by_relevance(user_question, df)["text"].values:
        this_text_tokens = len(tokenizer.encode(text))
        token_count += this_text_tokens
        if token_count <= token_limit:
            context_list.append(text)
        else:
            break
    
    # Build the prompt with the selected context and question
    prompt = prompt_template.format(
        "\n\n###\n\n".join(context_list),
        user_question
    )
    return prompt

## Custom Performance Demonstration

TODO: In the cells below, demonstrate the performance of your custom query using at least 2 questions. For each question, show the answer from a basic `Completion` model query as well as the answer from your custom query.

### Question 1

In [20]:
question1 = "What were the major fashion trends in 2023?"
#question2 = "What did fashion experts say about sustainable fashion in 2023?"
question2 = "What did fashion experts say about  fashion in 2023?"



In [21]:
df_copy1 = get_rows_sorted_by_relevance(question1, df)
df_copy2 = get_rows_sorted_by_relevance(question2, df)

In [22]:
original_prompt_first_question = """
Question: ""What were the major fashion trends in 2023?"
Answer:
"""
initial_question1_answer = openai.Completion.create(
    model="gpt-3.5-turbo-instruct",
    prompt=original_prompt_first_question,
    max_tokens=150
)["choices"][0]["text"].strip()
print(initial_question1_answer)

Unfortunately, it is impossible to accurately predict the fashion trends for a specific year, especially one that is so far in the future. Fashion is constantly evolving and trends can change quickly. Additionally, many unforeseen factors can influence fashion trends, such as cultural events, industry advancements, and societal preferences. It is best to stay updated on current fashion trends as they arise rather than trying to predict future ones.


In [23]:
custom_first_question_prompt = build_prompt(question1, df_copy1, tokenizer,2000)
COMPLETION_MODEL_NAME = "gpt-3.5-turbo-instruct"
response = openai.Completion.create(
    model=COMPLETION_MODEL_NAME,
    prompt=custom_first_question_prompt,
    max_tokens=150
)
answer = response["choices"][0]["text"].strip()
print(answer)

Some major fashion trends in 2023 included sheer and see-through clothing, red hues, denim, cobalt blue, elevated basics, daytime shine, maxi skirts, cargo pants, indie sleaze, minimalism, tailored looks, and more mesh details.


### Question 2

In [24]:
original_prompt_second_question = """
Question: ""What did fashion experts say about  fashion in 2023?"
Answer:
"""
initial_question2_answer = openai.Completion.create(
    model="gpt-3.5-turbo-instruct",
    prompt=original_prompt_second_question,
    max_tokens=150
)["choices"][0]["text"].strip()
print(initial_question2_answer)

There is no way to accurately predict what fashion experts will say about fashion trends in 2023 as it is still several years away. Trends and styles are constantly evolving and what may be popular in 2021 may not be in 2023. Fashion is also greatly influenced by current events, societal shifts, and cultural changes, making it difficult to make specific predictions so far in advance. It is best to stay up to date with current fashion trends and consumer behavior for a better understanding of what may be popular in 2023.


In [25]:
custom_second_question_prompt = build_prompt(question2, df_copy2, tokenizer,3000)


COMPLETION_MODEL_NAME = "gpt-3.5-turbo-instruct"
response = openai.Completion.create(
    model=COMPLETION_MODEL_NAME,
    prompt=custom_second_question_prompt,
    max_tokens=350
)
answer = response["choices"][0]["text"].strip()
print(answer)

Fashion experts identified several emerging trends for the spring/summer 2023 season, including a continuation of the maximalist trend, an emphasis on surrealism and 3D designs, a return to simplicity and everyday dressing, a focus on green and cobalt blue, and a resurgence of nostalgic and grunge-inspired styles. They also noted the prevalence of sheer fabrics and statement accessories, as well as the ongoing popularity of utility and cargo-inspired pieces.


In [26]:

'''
This demonstration highlights that the plain model, lacking access to recent or specific data, gives vague or non-informative answers. In contrast, the custom model, equipped with the 2023 fashion trends dataset, provides relevant, authoritative, and up-to-date responses, directly reflecting the information in the data. This shows the value of customizing language models with domain-specific knowledge for improved performance in specialized Q&A tasks.
'''

'\nThis demonstration highlights that the plain model, lacking access to recent or specific data, gives vague or non-informative answers. In contrast, the custom model, equipped with the 2023 fashion trends dataset, provides relevant, authoritative, and up-to-date responses, directly reflecting the information in the data. This shows the value of customizing language models with domain-specific knowledge for improved performance in specialized Q&A tasks.\n'